In [1]:
import random

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import numpy as np
from soupsieve import select

In [2]:
X, y = load_diabetes(return_X_y=True)
X.shape, y.shape

((442, 10), (442,))

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [4]:
sk = LinearRegression()

In [5]:
sk.fit(X_train, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [6]:
sk.intercept_, sk.coef_

(np.float64(151.34560453985995),
 array([  37.90402135, -241.96436231,  542.42875852,  347.70384391,
        -931.48884588,  518.06227698,  163.41998299,  275.31790158,
         736.1988589 ,   48.67065743]))

In [7]:
y_preds = sk.predict(X_test)
r2_score(y_test, y_preds)

0.4526027629719195

In [33]:
import random

class MiniBatchGDRegressor:
    def __init__(self, learning_rate: float = 0.001, epochs: int = 100, batch_size: int = 10):
        self.coef_ = None
        self.intercept_ = None
        self.lr = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)
        n_samples, n_features = X.shape

        self.intercept_ = 0.0
        self.coef_ = np.zeros(n_features)

        for _ in range(self.epochs):
            indices = np.random.permutation(n_samples)

            for start in range(0, n_samples, self.batch_size):
                end = start + self.batch_size
                batch_idx = indices[start:end]

                Xb = X[batch_idx]
                yb = y[batch_idx]

                y_pred = Xb @ self.coef_ + self.intercept_
                error = yb - y_pred

                B = len(batch_idx)

                intercept_gradient = (-2/B) * np.sum(error)
                coef_gradient = (-2/B) * (Xb.T @ error)

                self.intercept_ -= (self.lr * intercept_gradient)
                self.coef_ -= (self.lr * coef_gradient)

        return self

    def predict(self, X):
        X = np.asarray(X)
        return self.intercept_ + X @ self.coef_

In [62]:
mbgd = MiniBatchGDRegressor(learning_rate=0.01, epochs=1000, batch_size=15)

In [63]:
mbgd.fit(X_train, y_train)

In [64]:
mbgd.intercept_, mbgd.coef_

(np.float64(150.89969787702736),
 array([  53.86236923, -125.80623856,  412.84185343,  278.24007561,
         -22.69121592,  -65.70025798, -197.15342625,  148.93255737,
         316.67150243,  145.27500012]))

In [65]:
y_preds = mbgd.predict(X_test)
r2_score(y_test, y_preds)

0.45450615949945106

### Using SKlearn SGD for mini-batch

In [66]:
from sklearn.linear_model import SGDRegressor

In [67]:
sk = SGDRegressor(learning_rate='constant', eta0=0.01)

In [69]:
batch_size = 35

for i in range(100):
    idx = random.sample(range(X_train.shape[0]), batch_size)
    sk.partial_fit(X_train[idx], y_train[idx])

In [71]:
sk.intercept_, sk.coef_

(array([155.29561818]),
 array([ 23.05908531,  -4.432282  ,  68.13666858,  51.0474757 ,
         19.28597888,  12.51756021, -40.93311921,  44.27855551,
         63.79489187,  39.75682649]))

In [73]:
y_preds = sk.predict(X_test)
r2_score(y_test, y_preds)

0.16634761600298054